In [48]:
import itertools
import time


# ----------------------------------------------------------------------
# macOS keycodes
# ----------------------------------------------------------------------

KEYCODES = {
    "ENTER": 36,
    "DOWN": 125,
    "UP": 126,
}


# ----------------------------------------------------------------------
# Keyboard
# ----------------------------------------------------------------------

def press(key):
    """
    Send a key using the macOS keycode mapping.

    Example:
        press("ENTER")
        press("DOWN")
        press("UP")
    """
    send_key_code(KEYCODES[key])


# ----------------------------------------------------------------------
# Replace these with your existing implementations
# ----------------------------------------------------------------------

def screenshot(filename):
    """Take a screenshot using your existing mechanism."""
    pass


def memdump(filename):
    """Take a memory dump using your existing mechanism."""
    pass


def wait_for_screen(delay=0.5):
    """Optional delay / screen synchronization."""
    time.sleep(delay)


# ----------------------------------------------------------------------
# Menu navigation
# ----------------------------------------------------------------------

def select_option(option):
    """
    Select an option on a menu.

    Menus start with the first option selected.

    option is zero-based:
        0 = first option
        1 = second option
        2 = third option
        ...
    """

    for _ in range(option):
        press("DOWN")
        wait_for_screen()

    press("ENTER")
    wait_for_screen()


# ----------------------------------------------------------------------
# Mission generation
# ----------------------------------------------------------------------

def run_mission(
    theater,
    roe,
    mission_type,
    difficulty,
    landing_type,
    mission_number,
):
    """
    Run one complete mission configuration.

    All menu options are zero-based.
    """

    name = (
        f"{mission_number:03d}_"
        f"t{theater}_"
        f"r{roe}_"
        f"m{mission_type}_"
        f"d{difficulty}_"
        f"l{landing_type}"
    )

    print(f"Running {name}")

    # --------------------------------------------------------------
    # Pilot selection screen
    # --------------------------------------------------------------

    # Enter = select pilot and go to parameter screen 1
    press("ENTER")
    wait_for_screen(2)

    # --------------------------------------------------------------
    # Screen 0: Select change mission
    # --------------------------------------------------------------

    press("UP")
    wait_for_screen()
    press("ENTER")
    wait_for_screen()


    # --------------------------------------------------------------
    # Screen 1: Theater (4 options)
    # --------------------------------------------------------------
    
    select_option(theater)
    wait_for_screen()

    # --------------------------------------------------------------
    # Screen 2: Rules of engagement (3 options)
    # --------------------------------------------------------------

    select_option(roe)
    wait_for_screen()

    # --------------------------------------------------------------
    # Screen 3: Mission type (4 options)
    # --------------------------------------------------------------

    select_option(mission_type)
    wait_for_screen()

    # --------------------------------------------------------------
    # Screen 4: Opponent difficulty (4 options)
    # --------------------------------------------------------------

    select_option(difficulty)
    wait_for_screen()

    # --------------------------------------------------------------
    # Screen 5: Landing type (3 options)
    # --------------------------------------------------------------

    select_option(landing_type)
    wait_for_screen(6)

    # --------------------------------------------------------------
    # Briefing screen
    # --------------------------------------------------------------

    # Enter -> mission targets
    press("ENTER")
    wait_for_screen()

    screenshot(f"{name}_targets.png")

    # Enter -> return to briefing
    press("ENTER")
    wait_for_screen()

    # --------------------------------------------------------------
    # Flight plan
    # --------------------------------------------------------------

    # Up twice -> select flight plan
    press("UP")
    wait_for_screen()
    press("UP")
    wait_for_screen()

    # Enter -> view flight plan
    press("ENTER")
    wait_for_screen()

    screenshot(f"{name}_flightplan.png")

    # Enter -> return to briefing
    press("ENTER")
    wait_for_screen()

    # --------------------------------------------------------------
    # Memory dump
    # --------------------------------------------------------------

    memdump(f"{name}.bin")

    # --------------------------------------------------------------
    # Exit briefing room
    # --------------------------------------------------------------

    press("ENTER")
    wait_for_screen()

    # --------------------------------------------------------------
    # Arming complete
    # --------------------------------------------------------------

    press("ENTER")
    wait_for_screen(1)

    # --------------------------------------------------------------
    # Return to new mission
    # --------------------------------------------------------------

    press("UP")
    wait_for_screen()
    press("UP")
    wait_for_screen()
    press("UP")
    wait_for_screen()
    press("ENTER")
    wait_for_screen()

In [18]:
from debugbox import DOSBoxDebugger
from pathlib import Path
import time
from collections import Counter
import subprocess

In [42]:
appname='/opt/homebrew/bin/dosbox-x'

def activate(app):
    subprocess.run(["osascript", "-e", f'tell application "System Events" to set frontmost of process "{app}" to true'])
def send_key_code(code, app=appname, modifier=None):
    subprocess.run(["osascript", "-e", f'tell application "{appname}" to activate'])
    mod = f" using {{{modifier}}}" if modifier else ""
    script = f'''
      tell application "System Events"
      tell process "{app}"
          key code {code}{mod}
      end tell
      end tell'''
    subprocess.run(["osascript", "-e", script])

In [63]:
dbg = DOSBoxDebugger(
    conf_path="/Users/robmaxwell/Library/Preferences/f19-dosbox.conf",
    dump_path="/Users/robmaxwell/projects/f19/MEMDUMP.BIN",
)
dbg.start()

AUTOTYPE ENTER
LOG: Early LOG Init complete
LOG: CONFIG: Loaded config file: /Users/robmaxwell/Library/Preferences/f19-dosbox.conf
LOG: DOSBox-X's working directory: /Users/robmaxwell/projects/f19
LOG: Logging init: beginning logging proper. This is the end of the early init logging
LOG: Logging: opened logfile '/Users/robmaxwell/projects/f19/logs/f19-dosbox.log' successfully. All further logging will go to this file.
LOG: DOSBox-X version 2026.08.02 Commit e522642 (macOS ARM SDL2 64-bit)
LOG: macOS EXE path: /opt/homebrew/bin/dosbox-x
LOG: macOS Resource path: 
LOG: macOS: NSTouchBar support detected in system
LOG: Prevent capture: 0
LOG: SDL: version 2.32.70, Video cocoa, Audio coreaudio
LOG: Host keyboard layout is now us (US English)
LOG: Mapper keyboard layout is now us (US English)
LOG: SDL2 reports desktop display mode 1470 x 956
LOG: The default output for the video system: opengl
LOG: OpenGL Version : 2.1 Metal - 89.4
LOG: Configured windowposition: -
LOG: SDL: Current window 

In [51]:
def start_to_roster(bp):
    dbg.send(f"BP {bp}")
    dbg.run()
    send_key_code(45, app=appname)
    time.sleep(1)
    send_key_code(18, app=appname)
    for i in range(5):
        send_key_code(36, app=appname)
        time.sleep(1)

In [64]:
dbg = DOSBoxDebugger(
    conf_path="/Users/robmaxwell/Library/Preferences/f19-dosbox.conf",
    dump_path="/Users/robmaxwell/projects/f19/MEMDUMP.BIN",
)
dbg.start()

start_to_roster("1CE8:00000E80")
run_mission(0, 0, 0, 0, 0, 0)


B_
P_
 _
1_C_E_8_:_0_0_0_0_0_E_8_0_LOG: DEBUG: Set breakpoint at 1CE8:0E80

_                
R_
U_
N_
(Running)                                                                      

0814:00000100 BC2607              mov  sp,0726                                  0814:00000103 BB7300              mov  bx,0073                                  000001030000072613000000 LOG: pixratio 1.200, dw true, dh false
LOG: Aspect ratio: 640 x 480  xToY=1.333 yToX=0.750


In [62]:

bp = '1CE8:00000E80'
dbg.send('BPDEL *')
dbg.send(f"BP {bp}")
dbg.run()


 [m
 [m
 [m
 LOG: DEBUG: Set breakpoint at 1CE8:0E80

               
 [m
 [m
 mN_
(Running)                                                                      

1CE8:00001413 E9A80A              jmp  00001EBE ($+aa8)       (down)            00001EBE1
94000000

1CE8:00001413 E9A80A              jmp  00001EBE ($+aa8)       (down)            15D7
I-> _                                                                          0320000000014137054299266
>1.006333333[178

In [65]:
run_mission(0, 0, 0, 0, 0, 0)

Running 000_t0_r0_m0_d0_l0
CA DB E8 1C.29 4F E8 1C)O00 47 E8 1CG
..

1CE8:00000E80 B83200              mov  ax,0032                                  1CE8:00000E83 F76EF2              imul word [bp-0E]           ss:[DAA8]=0000    1CE8E86 8BF0si,ax
1CE8E88 037604add  si,[bp+04]ss:[DABA]=5C86
1CE8E8B 837C2E00cmp  word [si+2E],0000ds:[002E]=3838
1CE8E8F 741Dje   00000EAE ($+1d)(no jmp)
I-> _                                                                          202CE82CE82CE85C867D34  CS=1CE800000E80  C1 Z0 S1 O0 A100DAB60000000DA865144869620983.515333333


In [67]:
import itertools
import time
import os
import json
import tempfile
import subprocess

import numpy as np
from PIL import Image

In [68]:
CONF_PATH = "/Users/robmaxwell/Library/Preferences/f19-dosbox.conf"
DUMP_PATH = "/Users/robmaxwell/projects/f19/MEMDUMP.BIN"

SCREENSHOT_DIR = "/Users/robmaxwell/projects/f19/bp_search"

os.makedirs(SCREENSHOT_DIR, exist_ok=True)

In [69]:
def take_screenshot(path=None):
    """
    Take a screenshot of the current screen.

    Returns a PIL Image.
    """

    if path is None:
        fd, path = tempfile.mkstemp(suffix=".png")
        os.close(fd)
        delete_after = True
    else:
        delete_after = False

    try:
        subprocess.run(
            ["screencapture", "-x", path],
            check=True,
        )

        return Image.open(path).convert("RGB").copy()

    finally:
        if delete_after and os.path.exists(path):
            os.remove(path)

In [70]:
def screenshot_difference(img1, img2):
    """
    Return the fraction of pixels that differ.
    """

    a = np.asarray(img1, dtype=np.int16)
    b = np.asarray(img2, dtype=np.int16)

    if a.shape != b.shape:
        return 1.0

    diff = np.abs(a - b)

    changed = np.any(diff > 5, axis=2)

    return changed.mean()


def screenshots_match(img1, img2, threshold=0.001):
    """
    True if screenshots are sufficiently similar.
    """

    return screenshot_difference(img1, img2) <= threshold

In [71]:
def test_breakpoint(bp):
    """
    Start a fresh DOSBox instance, run to the supplied breakpoint,
    and capture the screen when the breakpoint fires.
    """

    print(f"Testing BP {bp}")

    dbg = DOSBoxDebugger(
        conf_path=CONF_PATH,
        dump_path=DUMP_PATH,
    )

    try:
        dbg.start()

        # Your existing function.
        # This should return when the breakpoint fires.
        start_to_roster(bp)

        print(f"  BP fired: {bp}")

        # Capture the screen while we're stopped
        img = take_screenshot()

        return img

    finally:
        # CHANGE THIS if your debugger uses a different shutdown method.
        dbg.quit()

In [72]:
def find_screen_groups(candidates):
    """
    Test every breakpoint and group breakpoints that produce
    the same screen.

    Returns a list of groups.
    """

    groups = []

    for i, bp in enumerate(candidates, start=1):

        print()
        print("=" * 60)
        print(f"[{i}/{len(candidates)}] {bp}")

        try:
            img = test_breakpoint(bp)

        except Exception as e:
            print(f"  ERROR: {e}")
            continue

        # See whether this screenshot matches an existing group
        matched_group = None

        for group in groups:

            difference = screenshot_difference(
                img,
                group["screenshot"],
            )

            if difference <= 0.001:
                matched_group = group
                break

        if matched_group is not None:

            matched_group["bps"].append(bp)

            print(
                f"  MATCHED SCREEN GROUP "
                f"{matched_group['id']}"
            )

        else:

            group = {
                "id": len(groups),
                "bps": [bp],
                "screenshot": img,
            }

            groups.append(group)

            print(
                f"  NEW SCREEN GROUP "
                f"{group['id']}"
            )

    return groups

In [75]:
candidates = [
    '2C2B:000007B0',
    '1CE8:00005FA8',
    '1CE8:00005FAD'
]
groups = find_screen_groups(candidates)


[1/3] 2C2B:000007B0
Testing BP 2C2B:000007B0
AUTOTYPE ENTER
LOG: Early LOG Init complete
LOG: CONFIG: Loaded config file: /Users/robmaxwell/Library/Preferences/f19-dosbox.conf
LOG: DOSBox-X's working directory: /Users/robmaxwell/projects/f19
LOG: Logging init: beginning logging proper. This is the end of the early init logging
LOG: Logging: opened logfile '/Users/robmaxwell/projects/f19/logs/f19-dosbox.log' successfully. All further logging will go to this file.
LOG: DOSBox-X version 2026.08.02 Commit e522642 (macOS ARM SDL2 64-bit)
LOG: macOS EXE path: /opt/homebrew/bin/dosbox-x
LOG: macOS Resource path: 
LOG: macOS: NSTouchBar support detected in system
LOG: Prevent capture: 0
LOG: SDL: version 2.32.70, Video cocoa, Audio coreaudio
LOG: Host keyboard layout is now us (US English)
LOG: Mapper keyboard layout is now us (US English)
LOG: SDL2 reports desktop display mode 1470 x 956
LOG: The default output for the video system: opengl
LOG: OpenGL Version : 2.1 Metal - 89.4
LOG: Configur